# Module 3 Project Solution: Detection, Segmentation, and Real-Time Pipeline Logic


> **Colab setup:** This solution notebook uses only the default open-source Python stack available in Google Colab: NumPy, Matplotlib, and scikit-learn. No `pip install`, no API keys, and no external authorization are required.
>
> Running all cells produces the complete reference output, including plots and metrics.


## Project map
This project uses synthetic images so the detection and segmentation logic is visible and fast.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

def make_rectangle_sample(h=64, w=64):
    img = np.zeros((h, w))
    mask = np.zeros((h, w), dtype=bool)
    y0, x0 = np.random.randint(8, 28, size=2)
    height, width = np.random.randint(14, 26, size=2)
    mask[y0:y0+height, x0:x0+width] = True
    img[mask] = 1.0
    img += np.random.normal(0, 0.12, size=img.shape)
    return np.clip(img, 0, 1), mask

img, mask = make_rectangle_sample()
plt.imshow(img, cmap="gray")
plt.title("Synthetic inspection image")
plt.axis("off")
plt.show()


In [ ]:
def bbox_from_mask(mask):
    ys, xs = np.where(mask)
    return np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])

bbox = bbox_from_mask(mask)
print("bbox:", bbox)

plt.imshow(img, cmap="gray")
x1, y1, x2, y2 = bbox
plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="red", linewidth=2))
plt.title("Bounding box from mask")
plt.axis("off")
plt.show()


In [ ]:
pred_mask = np.roll(mask, shift=4, axis=1)

def mask_iou(a, b):
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / union

def mask_dice(a, b):
    intersection = np.logical_and(a, b).sum()
    return 2 * intersection / (a.sum() + b.sum())

print("IoU:", mask_iou(pred_mask, mask))
print("Dice:", mask_dice(pred_mask, mask))


In [ ]:
boxes = np.array([
    [10, 10, 32, 34],
    [13, 12, 35, 36],
    [38, 15, 58, 35],
    [40, 17, 57, 34],
])
scores = np.array([0.92, 0.78, 0.88, 0.60])

def box_iou(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    area_box = (box[2]-box[0]) * (box[3]-box[1])
    area_boxes = (boxes[:, 2]-boxes[:, 0]) * (boxes[:, 3]-boxes[:, 1])
    return inter / (area_box + area_boxes - inter + 1e-9)

def nms(boxes, scores, threshold=0.4):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        ious = box_iou(boxes[i], boxes[order[1:]])
        order = order[1:][ious < threshold]
    return keep

keep = nms(boxes, scores)
print("kept indices:", keep)

plt.figure(figsize=(5, 5))
plt.xlim(0, 64); plt.ylim(64, 0); plt.grid(True)
for idx, b in enumerate(boxes):
    color = "green" if idx in keep else "gray"
    plt.gca().add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, color=color, linewidth=2))
    plt.text(b[0], b[1]-1, f"{idx}: {scores[idx]:.2f}", color=color)
plt.title("NMS result: green boxes kept")
plt.show()


In [ ]:
prob = img
threshold = 0.5
pred = prob > threshold

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("image")
axes[1].imshow(prob, cmap="viridis"); axes[1].set_title("probability-like map")
axes[2].imshow(pred, cmap="gray"); axes[2].set_title("threshold mask")
for ax in axes: ax.axis("off")
plt.show()
print("Dice after threshold:", mask_dice(pred, mask))
